# SemEval-2026 Task 13 — Subtask B: Improved Model

**Multi-Class Code Authorship Detection** (11 classes: human + 10 LLM families)

### Workflow:
1. Run setup cells (install, data, imports, class definitions)
2. **Step 1:** HP search (~50 min) — finds best learning rate + max_length
3. **Step 2:** Final training with best params (~30-40 min)
4. **Step 3:** Generate predictions on test set

**Runtime:** Set to `T4 GPU` via `Runtime → Change runtime type`

In [ ]:
!pip install --upgrade transformers datasets scikit-learn accelerate -q

## Data Setup
Uncomment the option matching your environment.

In [ ]:
# ============================================================
# OPTION A: Clone repo (for Google Colab)
# ============================================================
# !git clone https://github.com/mbzuai-nlp/SemEval-2026-Task13.git
# TRAIN_PATH = "SemEval-2026-Task13/task_b/task_b_training_set.parquet"
# VAL_PATH   = "SemEval-2026-Task13/task_b/task_b_validation_set.parquet"
# TEST_PATH  = "SemEval-2026-Task13/task_b/task_b_test_set_sample.parquet"

# ============================================================
# OPTION B: Kaggle input paths
# ============================================================
# TRAIN_PATH = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_training_set.parquet"
# VAL_PATH   = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_validation_set.parquet"
# TEST_PATH  = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_test_set_sample.parquet"

# ============================================================
# OPTION C: Download from HuggingFace
# ============================================================
from datasets import load_dataset
print("Downloading SemEval-2026-Task13 Subtask B from HuggingFace...")
hf_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "B")
hf_dataset['train'].to_parquet('task_b_train.parquet')
hf_dataset['validation'].to_parquet('task_b_val.parquet')
if 'test' in hf_dataset:
    hf_dataset['test'].to_parquet('task_b_test.parquet')
    TEST_PATH = 'task_b_test.parquet'
else:
    TEST_PATH = None
    print("No test split available on HuggingFace.")
TRAIN_PATH = 'task_b_train.parquet'
VAL_PATH   = 'task_b_val.parquet'
print("Done!")

# ============================================================
print(f"Train: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
print(f"Test:  {TEST_PATH}")

## Configuration

In [ ]:
USE_SUBSET = True            # False = full 500K dataset (slower)
HUMAN_SUBSET_SIZE = 20000    # Downsample human class to this
VAL_FRACTION = 0.2           # Keep 20% of validation

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
import gc
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import Dataset
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    f1_score
)
import warnings
warnings.filterwarnings("ignore")

# Check transformers version for API compatibility
TRANSFORMERS_VERSION = tuple(int(x) for x in transformers.__version__.split('.')[:2])
print(f"Transformers version: {transformers.__version__}")

ID_TO_LABEL = {
    0: "human", 1: "deepseek", 2: "qwen", 3: "01-ai", 4: "bigcode",
    5: "gemma", 6: "phi", 7: "meta-llama", 8: "ibm-granite", 9: "mistral", 10: "openai"
}
LABEL_TO_ID = {v: k for k, v in ID_TO_LABEL.items()}
NUM_LABELS = len(ID_TO_LABEL)

print(f"Task B: {NUM_LABELS}-class classification")
print(f"Labels: {list(ID_TO_LABEL.values())}")
print(f"Subset mode: {USE_SUBSET}")

## Model & Training Classes

In [ ]:
class WeightedTrainer(Trainer):
    """
    Custom Trainer with class-weighted CrossEntropyLoss.
    Uses sqrt-scaled weights to avoid extreme ratios.
    """

    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        if class_weights is not None:
            self.class_weights = class_weights.to(self.model.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fn = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fn = nn.CrossEntropyLoss()

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
def stratified_subsample(df, human_subset_size=20000, random_state=42):
    """Keep ALL minority samples, downsample human class."""
    human_df = df[df['label'] == 0]
    minority_df = df[df['label'] != 0]

    if len(human_df) > human_subset_size:
        human_df = human_df.sample(n=human_subset_size, random_state=random_state)

    result = pd.concat([human_df, minority_df], ignore_index=True)
    result = result.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print(f"  Subsampled: {len(df)} -> {len(result)} samples")
    print(f"  Human: {len(human_df)} | Minority (all 10 classes): {len(minority_df)}")
    return result


def compute_class_weights(label_counts, num_labels, method="sqrt"):
    """
    Compute class weights with sqrt scaling to prevent extreme ratios.
    Raw inverse frequency: 220:1 ratio -> unstable.
    sqrt scaling: ~15:1 ratio -> stable.
    """
    total = sum(label_counts.get(i, 1) for i in range(num_labels))
    weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 1)
        raw_weight = total / (num_labels * count)
        if method == "sqrt":
            w = np.sqrt(raw_weight)
        elif method == "log":
            w = np.log1p(raw_weight)
        else:
            w = raw_weight
        weights.append(w)

    min_w = min(weights)
    weights = [w / min_w for w in weights]
    return torch.FloatTensor(weights)

In [ ]:
class ImprovedCodeTrainer:
    def __init__(self, max_length=512, model_name="microsoft/unixcoder-base"):
        self.max_length = max_length
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.class_weights = None

    def load_and_prepare_data(self):
        try:
            df = pd.read_parquet(TRAIN_PATH)
            print(f"Dataset columns: {df.columns.tolist()}")
            print(f"Sample data:\n{df.head()}")

            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            df['label'] = df['label'].astype(int)

            unique_labels = sorted(df['label'].unique())
            print(f"\nUnique labels: {unique_labels}")

            print(f"\nFull dataset label distribution:")
            label_counts = df['label'].value_counts().sort_index()
            for label_id, count in label_counts.items():
                name = ID_TO_LABEL.get(label_id, f"unknown-{label_id}")
                print(f"  {label_id} ({name:>12s}): {count:>7d} ({count/len(df)*100:.1f}%)")

            if USE_SUBSET:
                print(f"\n--- SUBSET MODE ---")
                df = stratified_subsample(df, human_subset_size=HUMAN_SUBSET_SIZE)

            label_counts = df['label'].value_counts().sort_index()
            self.class_weights = compute_class_weights(label_counts, NUM_LABELS, method="sqrt")

            print(f"\nClass weights (sqrt-scaled):")
            for i, w in enumerate(self.class_weights.tolist()):
                name = ID_TO_LABEL.get(i, f"unknown-{i}")
                count = label_counts.get(i, 0)
                print(f"  {name:>12s}: weight={w:>5.2f}  (n={count})")

            val_df = pd.read_parquet(VAL_PATH)
            val_df['label'] = val_df['label'].astype(int)

            if USE_SUBSET:
                print(f"\n--- Subsampling validation to {VAL_FRACTION*100:.0f}% ---")
                val_df = val_df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(frac=VAL_FRACTION, random_state=42)
                ).reset_index(drop=True)

            print(f"\nFinal -> Train: {len(df)} | Val: {len(val_df)}")
            return df, val_df

        except Exception as e:
            print(f"Error loading dataset: {e}")
            raise

    def initialize_model_and_tokenizer(self):
        print(f"\nInitializing {self.model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=NUM_LABELS,
            problem_type="single_label_classification"
        ).to('cuda')
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"Model loaded: {total_params/1e6:.1f}M params")

    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

    def prepare_datasets(self, train_df, val_df):
        print("\nTokenizing datasets...")
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset = Dataset.from_pandas(val_df[['code', 'label']])

        train_dataset = train_dataset.map(self.tokenize_function, batched=True, remove_columns=['code'])
        val_dataset = val_dataset.map(self.tokenize_function, batched=True, remove_columns=['code'])

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset = val_dataset.rename_column('label', 'labels')

        print(f"Done. Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        return train_dataset, val_dataset

    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        preds = np.argmax(predictions, axis=1)

        unique, counts = np.unique(preds, return_counts=True)
        pred_dist = {int(u): int(c) for u, c in zip(unique, counts)}
        print(f"  Prediction distribution: {pred_dist}")

        accuracy = accuracy_score(labels, preds)
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        weighted_f1 = f1_score(labels, preds, average='weighted', zero_division=0)

        return {
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'accuracy': accuracy,
        }

    def train(self, train_dataset, val_dataset, output_dir="./results",
              num_epochs=3, batch_size=16, learning_rate=2e-5):
        print("\n" + "="*60)
        print("STARTING TRAINING")
        print("="*60)

        steps_per_epoch = len(train_dataset) // (batch_size * 2)
        eval_save_steps = max(100, steps_per_epoch // 3)

        # Build TrainingArguments with version-compatible parameter names
        train_args_dict = dict(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            gradient_accumulation_steps=2,
            warmup_ratio=0.1,
            weight_decay=0.01,
            logging_steps=50,
            save_strategy="steps",
            save_steps=eval_save_steps,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="cosine",
            save_total_limit=2,
            fp16=True,
            report_to="none",
        )

        # Handle eval_strategy vs evaluation_strategy across versions
        try:
            train_args_dict['eval_strategy'] = 'steps'
            train_args_dict['eval_steps'] = eval_save_steps
            training_args = TrainingArguments(**train_args_dict)
        except TypeError:
            del train_args_dict['eval_strategy']
            train_args_dict['evaluation_strategy'] = 'steps'
            train_args_dict['eval_steps'] = eval_save_steps
            training_args = TrainingArguments(**train_args_dict)

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        # Handle tokenizer vs processing_class across versions
        trainer_kwargs = dict(
            class_weights=self.class_weights,
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )
        try:
            trainer = WeightedTrainer(processing_class=self.tokenizer, **trainer_kwargs)
        except TypeError:
            trainer = WeightedTrainer(tokenizer=self.tokenizer, **trainer_kwargs)

        print(f"Model: {self.model_name}")
        print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        print(f"Batch: {batch_size}x2, LR: {learning_rate}, MaxLen: {self.max_length}")
        print(f"Epochs: {num_epochs}, Eval every {eval_save_steps} steps")
        print(f"fp16: True, Scheduler: cosine")
        print()

        trainer.train()
        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        print(f"\nModel saved to {output_dir}")
        return trainer

    def evaluate_model(self, trainer, val_dataset):
        print("\n" + "="*60)
        print("EVALUATION")
        print("="*60)
        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids

        unique, counts = np.unique(y_pred, return_counts=True)
        print("\nPrediction distribution:")
        for u, c in zip(unique, counts):
            name = ID_TO_LABEL.get(int(u), f"unknown-{u}")
            print(f"  Predicted {name:>12s} (label {u}): {c} times")

        target_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
        print("\nPer-Class Classification Report:")
        print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        print(f"\n>>> COMPETITION METRIC (Macro F1): {macro_f1:.4f} <<<")
        return predictions

    def run_full_pipeline(self, output_dir="./results", num_epochs=3,
                          batch_size=16, learning_rate=2e-5):
        try:
            train_df, val_df = self.load_and_prepare_data()
            self.initialize_model_and_tokenizer()
            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)
            trainer = self.train(
                train_dataset, val_dataset,
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate
            )
            self.evaluate_model(trainer, val_dataset)
            print("\nPipeline completed successfully!")
            return trainer
        except Exception as e:
            print(f"Error in pipeline: {e}")
            raise

---
## Step 1: Quick HP Search (~50 min on T4)

Tests 6 combos: learning_rate × max_length, 2 epochs each.

**Skip this cell** if you want to go straight to training with defaults.

In [ ]:
SEARCH_GRID = {
    'learning_rate': [1e-5, 2e-5, 5e-5],
    'max_length': [128, 256],
}
SEARCH_EPOCHS = 2

results = []

for lr in SEARCH_GRID['learning_rate']:
    for ml in SEARCH_GRID['max_length']:
        print(f"\n{'#'*60}")
        print(f"TRIAL: lr={lr}, max_length={ml}")
        print(f"{'#'*60}")

        trial_obj = ImprovedCodeTrainer(
            max_length=ml,
            model_name="microsoft/unixcoder-base",
        )

        try:
            trial_trainer = trial_obj.run_full_pipeline(
                output_dir=f"hp_search_lr{lr}_ml{ml}",
                num_epochs=SEARCH_EPOCHS,
                batch_size=16,
                learning_rate=lr
            )

            eval_results = [x for x in trial_trainer.state.log_history if 'eval_macro_f1' in x]
            best_f1 = max([x['eval_macro_f1'] for x in eval_results]) if eval_results else 0.0

            results.append({'learning_rate': lr, 'max_length': ml, 'best_macro_f1': best_f1})
            print(f"\n>>> Trial result: Macro F1 = {best_f1:.4f}")

        except Exception as e:
            print(f"Trial failed: {e}")
            results.append({'learning_rate': lr, 'max_length': ml, 'best_macro_f1': 0.0})

        del trial_obj
        try: del trial_trainer
        except: pass
        gc.collect()
        torch.cuda.empty_cache()

print("\n" + "="*60)
print("HYPERPARAMETER SEARCH RESULTS")
print("="*60)
results_df = pd.DataFrame(results).sort_values('best_macro_f1', ascending=False)
print(results_df.to_string(index=False))

best = results_df.iloc[0]
BEST_LR = best['learning_rate']
BEST_ML = int(best['max_length'])
print(f"\n>>> BEST: lr={BEST_LR}, max_length={BEST_ML}, Macro F1={best['best_macro_f1']:.4f}")

---
## Step 2: Final Training

If you skipped HP search, uncomment the manual lines below.

In [ ]:
# If you skipped HP search, set manually:
# BEST_LR = 2e-5
# BEST_ML = 256

OUTPUT_DIR = "taskB-improved-model"

gc.collect()
torch.cuda.empty_cache()

trainer_obj = ImprovedCodeTrainer(
    max_length=BEST_ML,
    model_name="microsoft/unixcoder-base",
)

trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=10,
    batch_size=16,
    learning_rate=BEST_LR
)

---
## Step 3: Predict on Test Set

In [ ]:
import torch
from itertools import chain
from datasets import load_dataset
from tqdm import tqdm

@torch.no_grad()
def predict_with_trainer(trainer_obj, parquet_path, output_path,
                         max_length=512, batch_size=16, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = trainer_obj.model
    tokenizer = trainer_obj.tokenizer
    if tokenizer is None:
        raise ValueError("trainer_obj must have a tokenizer.")

    model.to(device)
    model.eval()

    ds = load_dataset("parquet", data_files=parquet_path, split="train", streaming=True)
    it = iter(ds)
    first = next(it)
    if not {"ID", "code"}.issubset(first.keys()):
        raise ValueError("Parquet must contain 'ID' and 'code' columns")
    stream = chain([first], it)

    def batcher(iterator, bs):
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    with open(output_path, "w") as f:
        f.write("ID,prediction\n")
        for batch in tqdm(batcher(stream, batch_size), desc="Predicting"):
            codes = [row["code"] for row in batch]
            ids   = [row["ID"] for row in batch]
            enc = tokenizer(codes, truncation=True, padding=True,
                           max_length=max_length, return_tensors="pt")
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            pred_labels = logits.argmax(dim=-1).cpu().tolist()
            for ex_id, pred in zip(ids, pred_labels):
                f.write(f"{ex_id},{pred}\n")

    print(f"Predictions saved to {output_path}")

In [ ]:
if TEST_PATH is not None:
    OUT_CSV = "submission_b.csv"
    predict_with_trainer(
        trainer_obj=trainer_obj,
        parquet_path=TEST_PATH,
        output_path=OUT_CSV,
        max_length=BEST_ML,
        batch_size=32,
        device="cuda"
    )
    print("Wrote:", OUT_CSV)
else:
    print("No test path set. Update TEST_PATH in the data setup cell.")

In [ ]:
# Download submission (Colab only)
# from google.colab import files
# files.download(OUT_CSV)